# Strategy Research: Volatility-Scaled Momentum (Core)

| Field | Value |
|-------|-------|
| **Researcher** | Elena |
| **Date** | 2026-03-15 |
| **Round** | R1 |
| **Asset Class** | Equity (US Large Cap) |
| **Strategy Folder** | `research/strategies/vol_scaled_momentum_2026-03-13_conditional/` |

**Hypothesis:** Stocks with low recent realized volatility and strong risk-adjusted momentum deliver superior risk-adjusted returns due to the low-volatility anomaly and momentum persistence, while a mean-reversion dampener reduces crash exposure from overextended names.

## 1. Hypothesis & Literature Review

### Economic Rationale

This strategy combines two of the most robust cross-sectional anomalies in equity markets: the low-volatility anomaly and momentum. The low-volatility anomaly, documented by Frazzini & Pedersen (2014) in "Betting Against Beta," shows that low-beta (low-vol) stocks outperform on a risk-adjusted basis because leverage-constrained investors overpay for volatile stocks. Simultaneously, Jegadeesh & Titman (1993) established that stocks with strong past returns continue to outperform over 3-12 month horizons due to investor underreaction and gradual information diffusion.

By scaling momentum by volatility (computing a Sharpe-like signal), we tilt toward stocks that deliver high returns per unit of risk -- the intersection of momentum and quality. The mean-reversion dampener (negative weight on 20-day z-score) penalizes stocks that have run too far too fast, reducing exposure to the left-tail crash risk that plagues naive momentum strategies (Daniel & Moskowitz, 2016).

### Who Loses Money?

Leverage-constrained investors (pension funds, mutual funds) who systematically overpay for high-beta stocks provide the low-volatility premium. Behavioral underreaction by investors who anchor on stale information provides the momentum premium. Market makers who sell liquidity to momentum traders absorb the mean-reversion dampener costs.

### Economic Mechanism

The edge persists because: (1) leverage constraints are structural for major institutional investors (regulatory capital requirements), (2) behavioral biases (anchoring, disposition effect) are deeply rooted in human psychology, and (3) the combination of factors reduces crowding in any single factor.

### Academic Citations

1. Frazzini & Pedersen (2014), "Betting Against Beta," *Journal of Financial Economics*. Low-beta stocks outperform risk-adjusted across 20+ markets over 50+ years.
2. Jegadeesh & Titman (1993), "Returns to Buying Winners and Selling Losers," *Journal of Finance*. Seminal momentum paper; 12-1 momentum earns 1%/month cross-sectionally.
3. Blitz & van Vliet (2007), "The Volatility Effect," *Journal of Portfolio Management*. Low-volatility stocks earn higher risk-adjusted returns globally.
4. Daniel & Moskowitz (2016), "Momentum Crashes," *Journal of Financial Economics*. Momentum strategies experience severe crashes; volatility scaling mitigates this.
5. Asness, Moskowitz & Pedersen (2013), *"Value and Momentum Everywhere"*, *Journal of Finance*. Combining momentum with other factors produces superior Sharpe ratios.

### Key Risks

- **Momentum crash risk:** Sudden momentum reversals (e.g., March 2009) can cause large drawdowns.
- **Factor crowding:** Low-vol and momentum are well-known; alpha may be reduced by institutional adoption.
- **Concentration risk:** 30-stock universe may lead to sector concentration and idiosyncratic exposure.

In [1]:
# Cell 3: Setup & Imports
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Project root
PROJECT_ROOT = '/Users/zelin/Desktop/PA Investment/Invest_strategy'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# === STRATEGY CONFIGURATION (pre-committed, do NOT change after backtest) ===
STRATEGY_NAME = 'vol_scaled_momentum_core'
TICKERS = [
    'AAPL','MSFT','GOOGL','AMZN','META','BRK-B','JPM','JNJ','UNH','XOM',
    'CVX','PG','V','MA','HD','BAC','WMT','KO','PEP','ABBV',
    'MRK','TMO','COST','AVGO','CSCO','TXN','ACN','LIN','NEE','MDT'
]
START_DATE = '2010-01-01'
END_DATE = '2024-12-31'
IS_END = '2021-12-31'   # In-sample ends here
OOS_START = '2022-01-01' # Out-of-sample starts here

# Pre-committed signal weights (from proposal.md -- DO NOT TUNE)
WEIGHT_INVVOL = 0.40    # VolatilitySignal(lookback=60)
WEIGHT_MOMENTUM = 0.40  # MomentumSignal(lookback=252, skip=21) -> sharpe-scaled
WEIGHT_MEANREV = -0.20  # MeanReversionSignal(lookback=20)

# Pre-committed optimization parameters
RISK_AVERSION = 3.0
MAX_WEIGHT = 0.12
MIN_WEIGHT = 0.0   # Long-only
REBALANCE_FREQ = '2M'  # Bi-monthly

# Cost assumptions
FIXED_COST_BPS = 5.0    # $0.005/share approximation
PROP_COST_BPS = 10.0    # 10 bps proportional

# Number of trials (honest count -- only 1 pre-committed config)
N_TRIALS = 1

print(f'Strategy: {STRATEGY_NAME}')
print(f'Universe: {len(TICKERS)} tickers')
print(f'Period: {START_DATE} to {END_DATE}')
print(f'IS/OOS split: {IS_END} | {OOS_START}')
print(f'Signal weights: InvVol={WEIGHT_INVVOL}, Mom={WEIGHT_MOMENTUM}, MeanRev={WEIGHT_MEANREV}')
print(f'Cost assumption: fixed={FIXED_COST_BPS}bps + proportional={PROP_COST_BPS}bps')

Strategy: vol_scaled_momentum_core
Universe: 30 tickers
Period: 2010-01-01 to 2024-12-31
IS/OOS split: 2021-12-31 | 2022-01-01
Signal weights: InvVol=0.4, Mom=0.4, MeanRev=-0.2
Cost assumption: fixed=5.0bps + proportional=10.0bps


In [2]:
# Cell 4: Data Loading & Inspection
import yfinance as yf

print('Downloading price data from yfinance...')
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=True)

# Extract close prices
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close']
else:
    prices = raw[['Close']]

# Ensure column names are clean strings
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(-1)

# Forward-fill gaps (max 5 days), then drop tickers with >5% missing
prices = prices.ffill(limit=5)
missing_pct = prices.isnull().mean()
good_tickers = missing_pct[missing_pct < 0.05].index.tolist()
prices = prices[good_tickers].dropna()

print(f'\nShape: {prices.shape}')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Trading days: {len(prices)}')
print(f'Tickers retained: {len(prices.columns)} / {len(TICKERS)}')
print(f'\nMissing data after cleaning:')
print(prices.isnull().sum().to_string())
print(f'\nReturn statistics (annualized):')
rets = prices.pct_change().dropna()
ann_ret = rets.mean() * 252
ann_vol = rets.std() * np.sqrt(252)
summary = pd.DataFrame({'Ann Return': ann_ret, 'Ann Vol': ann_vol, 'Sharpe': ann_ret / ann_vol})
print(summary.round(3).to_string())

[                       0%                       ]

[***                    7%                       ]  2 of 30 completed

[*****                 10%                       ]  3 of 30 completed

[********              17%                       ]  5 of 30 completed

[**********            20%                       ]  6 of 30 completed

...

[**********************87%*****************      ]  26 of 30 completed

[**********************93%********************   ]  28 of 30 completed

[**********************97%********************** ]  29 of 30 completed

[*********************100%***********************]  30 of 30 completed


Shape: (3773, 28)
Date range: 2010-01-04 to 2024-12-30
Trading days: 3773
Tickers retained: 28 / 30

Missing data after cleaning:
Ticker
AAPL     0
ACN      0
AMZN     0
AVGO     0
BAC      0
BRK-B    0
COST     0
CSCO     0
CVX      0
GOOGL    0
HD       0
JNJ      0
JPM      0
KO       0
LIN      0
MA       0
MDT      0
MRK      0
MSFT     0
NEE      0
PEP      0
PG       0
TMO      0
TXN      0
UNH      0
V        0
WMT      0
XOM      0

Return statistics (annualized):
        Ann Return  Ann Vol  Sharpe
Ticker                             
AAPL         0.284    0.279   1.019
ACN          0.190    0.243   0.783
AMZN         0.287    0.327   0.878
AVGO         0.413    0.368   1.123
BAC          0.141    0.336   0.419
BRK-B        0.147    0.195   0.757
COST         0.226    0.202   1.120
CSCO         0.119    0.259   0.461
CVX          0.115    0.268   0.431
GOOGL        0.205    0.273   0.749
HD           0.225    0.232   0.971
JNJ          0.097    0.167   0.578
JPM          0.17

In [3]:
# Cell 5: Signal Construction
# Using project signal classes with expanding-window z-score normalization
from alpha_research.backtests.strategies.signals import VolatilitySignal, MomentumSignal, MeanReversionSignal

# --- Signal 1: Inverse Volatility (lookback=60) ---
# VolatilitySignal already returns -vol, so higher value = lower vol
vol_signal = VolatilitySignal(lookback=60)
raw_vol = vol_signal.compute(prices)  # Returns -vol (negative = high vol)
print(f'Inverse Vol signal shape: {raw_vol.shape}')

# --- Signal 2: Risk-Adjusted Momentum (Sharpe-like: 12-1 momentum / trailing vol) ---
mom_signal = MomentumSignal(lookback=252, skip=21)
raw_mom = mom_signal.compute(prices)  # Raw 12-1 momentum returns

# Convert to Sharpe-like ratio: momentum / trailing 252d vol
trailing_vol = prices.pct_change().rolling(252, min_periods=252).std() * np.sqrt(252)
raw_sharpe_mom = raw_mom / trailing_vol.replace(0, np.nan)
raw_sharpe_mom = raw_sharpe_mom.replace([np.inf, -np.inf], np.nan)
print(f'Sharpe-Momentum signal shape: {raw_sharpe_mom.shape}')

# --- Signal 3: Mean Reversion Dampener (lookback=20) ---
mr_signal = MeanReversionSignal(lookback=20)
raw_mr = mr_signal.compute(prices)  # Negative z-score: high price vs MA = negative
print(f'Mean Reversion signal shape: {raw_mr.shape}')

# --- Expanding-window cross-sectional z-score normalization ---
# At each date, z-score across tickers using only expanding (past) data
MIN_PERIODS_ZSCORE = 60  # Minimum periods for expanding z-score

def expanding_zscore(signal_df, min_periods=MIN_PERIODS_ZSCORE):
    """Cross-sectional z-score at each date using expanding window."""
    # First: cross-sectional z-score (across tickers at each date)
    cs_mean = signal_df.mean(axis=1)
    cs_std = signal_df.std(axis=1)
    cs_zscore = signal_df.sub(cs_mean, axis=0).div(cs_std.replace(0, np.nan), axis=0)
    
    # Then: time-series expanding normalization to avoid look-ahead
    exp_mean = cs_zscore.expanding(min_periods=min_periods).mean()
    exp_std = cs_zscore.expanding(min_periods=min_periods).std()
    normalized = (cs_zscore - exp_mean) / exp_std.replace(0, np.nan)
    return normalized

z_vol = expanding_zscore(raw_vol)
z_mom = expanding_zscore(raw_sharpe_mom)
z_mr = expanding_zscore(raw_mr)

# Warmup: require at least 252 + 60 days of data before any signal is valid
WARMUP = 252 + MIN_PERIODS_ZSCORE
print(f'\nWarmup period: {WARMUP} trading days')
print(f'First valid signal date (approx): {prices.index[WARMUP].date()}')

# Verify no look-ahead: signals should be NaN before warmup
print(f'\nNaN counts in first {WARMUP} rows:')
print(f'  z_vol:  {z_vol.iloc[:WARMUP].isnull().all(axis=1).sum()} / {WARMUP} rows all-NaN')
print(f'  z_mom:  {z_mom.iloc[:WARMUP].isnull().all(axis=1).sum()} / {WARMUP} rows all-NaN')
print(f'  z_mr:   {z_mr.iloc[:WARMUP].isnull().all(axis=1).sum()} / {WARMUP} rows all-NaN')

Inverse Vol signal shape: (3773, 28)
Sharpe-Momentum signal shape: (3773, 28)
Mean Reversion signal shape: (3773, 28)

Warmup period: 312 trading days
First valid signal date (approx): 2011-03-30

NaN counts in first 312 rows:
  z_vol:  119 / 312 rows all-NaN
  z_mom:  311 / 312 rows all-NaN
  z_mr:   78 / 312 rows all-NaN


In [4]:
# Cell 6: Alpha Blending with Pre-Committed Weights

# Blend signals: alpha = 0.4 * z(-vol) + 0.4 * z(sharpe_mom) + (-0.2) * z(mr)
# Note: z_vol already represents -vol (low vol = positive signal)
# z_mr is already negative when price is extended above MA
# WEIGHT_MEANREV is -0.20, so we penalize extended stocks (double negative -> positive dampening)

alpha = (WEIGHT_INVVOL * z_vol + 
         WEIGHT_MOMENTUM * z_mom + 
         WEIGHT_MEANREV * z_mr)

# Drop warmup period
alpha = alpha.iloc[WARMUP:]

print(f'Alpha signal shape: {alpha.shape}')
print(f'Alpha date range: {alpha.index[0].date()} to {alpha.index[-1].date()}')
print(f'\nAlpha cross-sectional stats (time-series avg):')
print(f'  Mean:  {alpha.mean(axis=1).mean():.4f}')
print(f'  Std:   {alpha.std(axis=1).mean():.4f}')
print(f'  Skew:  {alpha.apply(lambda x: x.dropna().skew(), axis=1).mean():.4f}')
print(f'\nPre-committed weights: InvVol={WEIGHT_INVVOL}, Mom={WEIGHT_MOMENTUM}, MeanRev={WEIGHT_MEANREV}')
print('IMPORTANT: These weights were set BEFORE seeing any backtest results.')

# Plot average alpha rank over time
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
z_vol.iloc[WARMUP:].mean(axis=1).plot(ax=axes[0], alpha=0.7, label='Inv. Vol Z-Score')
axes[0].set_title('Average Cross-Sectional Z-Score: Inverse Volatility')
axes[0].axhline(0, color='black', linewidth=0.5)

z_mom.iloc[WARMUP:].mean(axis=1).plot(ax=axes[1], alpha=0.7, color='green', label='Sharpe-Mom Z-Score')
axes[1].set_title('Average Cross-Sectional Z-Score: Risk-Adjusted Momentum')
axes[1].axhline(0, color='black', linewidth=0.5)

alpha.mean(axis=1).plot(ax=axes[2], alpha=0.7, color='purple', label='Blended Alpha')
axes[2].set_title('Average Blended Alpha Signal')
axes[2].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(PROJECT_ROOT + '/research/strategies/vol_scaled_momentum_2026-03-13_conditional/signals_plot.png', dpi=100)
plt.show()
print('Signal plots saved.')

Alpha signal shape: (3461, 28)
Alpha date range: 2011-03-30 to 2024-12-30

Alpha cross-sectional stats (time-series avg):
  Mean:  -0.0148
  Std:   0.7453
  Skew:  -0.2649

Pre-committed weights: InvVol=0.4, Mom=0.4, MeanRev=-0.2
IMPORTANT: These weights were set BEFORE seeing any backtest results.


Signal plots saved.


In [5]:
# Cell 7: Portfolio Optimization Config + Backtest
from alpha_research.backtests.builder import PortfolioBuilder, PortfolioConfig
from alpha_research.backtests.costs import CompositeCostModel, ProportionalCostModel, FixedCostModel

# --- Build the PortfolioBuilder ---
config = PortfolioConfig(
    universe=list(prices.columns),
    signals=[],  # We supply alpha directly via signals dict
    optimization='mean_variance',
    risk_aversion=RISK_AVERSION,
    max_weight=MAX_WEIGHT,
    min_weight=MIN_WEIGHT,  # Long-only
    target_gross=1.0,
    rebalance_frequency='2M',  # Bi-monthly
    turnover_penalty=0.5,  # From proposal
    initial_cash=100000,
    commission=0.001,
)

builder = PortfolioBuilder(config=config)

# Inject price data directly (skip yfinance download inside builder)
builder.prices = prices

# Inject our pre-computed alpha as the signal the builder uses for optimization
# The builder expects signals as a dict of signal_name -> DataFrame
builder.signals = {'vol_scaled_momentum': alpha}

# Set initial equal weights -- dynamic_reoptimize=True will re-optimize
# at each rebalance using _optimize_weights_as_of() which correctly
# extracts per-asset alpha from the multi-column signal DataFrame
n_assets = len(prices.columns)
builder.weights = pd.Series(1.0 / n_assets, index=prices.columns)
print(f'Initial weights: equal-weight ({1.0/n_assets:.4f} each across {n_assets} assets)')
print('Dynamic reoptimization will compute optimal weights at each rebalance.')

# --- Cost model ---
cost_model = CompositeCostModel(
    models=(
        FixedCostModel(cost_per_trade=0.005),  # $0.005/share equivalent
        ProportionalCostModel(cost_bps=PROP_COST_BPS),  # 10 bps
    )
)

# --- Run full-sample backtest with dynamic reoptimization ---
# Start after warmup period
backtest_start = str(prices.index[WARMUP].date())

print(f'\nRunning backtest: {backtest_start} to {END_DATE}')
print(f'Dynamic reoptimize: True | Rebalance: bi-monthly | Cost model: Composite')

result = builder.backtest(
    start_date=backtest_start,
    end_date=END_DATE,
    cost_model=cost_model,
    dynamic_reoptimize=True,
)

print(f'\nBacktest complete. Days: {result["n_days"]}')

Initial weights: equal-weight (0.0357 each across 28 assets)
Dynamic reoptimization will compute optimal weights at each rebalance.

Running backtest: 2011-03-30 to 2024-12-31
Dynamic reoptimize: True | Rebalance: bi-monthly | Cost model: Composite



Backtest complete. Days: 3460


In [6]:
# Cell 8: Core Metrics Table

daily_returns = result['daily_returns']
equity_curve = result['equity_curve']

# Core metrics
total_return = result['total_return']
ann_return = result['annualized_return']
volatility = result['volatility']
sharpe = result['sharpe_ratio']
max_dd = result['max_drawdown']
n_days = result['n_days']

# Calmar ratio
calmar = ann_return / abs(max_dd) if max_dd != 0 else 0

# Sortino ratio
downside_returns = daily_returns[daily_returns < 0]
downside_vol = downside_returns.std() * np.sqrt(252)
sortino = (daily_returns.mean() * 252) / downside_vol if downside_vol > 0 else 0

# Annual turnover (approximate from weight changes)
# This is already computed inside the builder -- estimate from daily returns
# Turnover = sum of absolute position changes per year
annual_turnover = daily_returns.diff().abs().sum() / (n_days / 252) * 100

print('=' * 60)
print('CORE PERFORMANCE METRICS')
print('=' * 60)
print(f'  Total Return:       {total_return:.2%}')
print(f'  Annualized Return:  {ann_return:.2%}')
print(f'  Annualized Vol:     {volatility:.2%}')
print(f'  Sharpe Ratio:       {sharpe:.3f}')
print(f'  Sortino Ratio:      {sortino:.3f}')
print(f'  Max Drawdown:       {max_dd:.2%}')
print(f'  Calmar Ratio:       {calmar:.3f}')
print(f'  Trading Days:       {n_days}')
print(f'  Backtest Years:     {n_days/252:.1f}')

# Equity curve plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

equity_curve.set_index('date')['portfolio_value'].plot(ax=axes[0], linewidth=1.5)
axes[0].set_title(f'{STRATEGY_NAME} -- Equity Curve ({backtest_start} to {END_DATE})')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].grid(True, alpha=0.3)

# Drawdown plot
cum = (1 + daily_returns).cumprod()
dd = (cum - cum.cummax()) / cum.cummax()
dd.plot(ax=axes[1], color='red', linewidth=1)
axes[1].fill_between(dd.index, dd.values, 0, color='red', alpha=0.3)
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT + '/research/strategies/vol_scaled_momentum_2026-03-13_conditional/equity_curve.png', dpi=100)
plt.show()

CORE PERFORMANCE METRICS
  Total Return:       616.44%
  Annualized Return:  15.42%
  Annualized Vol:     18.06%
  Sharpe Ratio:       0.885
  Sortino Ratio:      1.121
  Max Drawdown:       -32.00%
  Calmar Ratio:       0.482
  Trading Days:       3460
  Backtest Years:     13.7


In [7]:
# Cell 9: PSR (Probabilistic Sharpe Ratio)
from alpha_research.backtests.stats.sharpe_tests import probabilistic_sharpe_ratio, sharpe_confidence_interval

psr = probabilistic_sharpe_ratio(daily_returns.values, benchmark_sharpe=0.0)
sr_ci = sharpe_confidence_interval(daily_returns.values, confidence=0.95)

print('=' * 60)
print('PROBABILISTIC SHARPE RATIO')
print('=' * 60)
print(f'  PSR (vs SR=0):       {psr:.4f} ({psr:.1%})')
print(f'  Gate threshold:      > 0.80')
print(f'  Gate status:         {"PASS" if psr > 0.80 else "FAIL"}')
print(f'')
print(f'  Sharpe 95% CI:       [{sr_ci[0]:.3f}, {sr_ci[2]:.3f}]')
print(f'  Sharpe point est:    {sr_ci[1]:.3f}')
print(f'')
print(f'  Interpretation: There is a {psr:.1%} probability that the true Sharpe')
print(f'  exceeds 0, accounting for skewness and kurtosis of the return distribution.')

PROBABILISTIC SHARPE RATIO
  PSR (vs SR=0):       0.9994 (99.9%)
  Gate threshold:      > 0.80
  Gate status:         PASS

  Sharpe 95% CI:       [0.420, 1.427]
  Sharpe point est:    0.885

  Interpretation: There is a 99.9% probability that the true Sharpe
  exceeds 0, accounting for skewness and kurtosis of the return distribution.


In [8]:
# Cell 10: Deflated Sharpe Ratio + MinBTL
from alpha_research.backtests.stats.sharpe_tests import deflated_sharpe_ratio
from alpha_research.backtests.stats.minimum_backtest import minimum_backtest_length
from scipy.stats import skew, kurtosis

# Deflated Sharpe (adjusts for N_TRIALS tested)
dsr = deflated_sharpe_ratio(daily_returns.values, n_trials=N_TRIALS)

# MinBTL: minimum backtest length for this Sharpe to be significant
obs_sharpe = sharpe  # annualized
ret_skew = float(skew(daily_returns.dropna().values))
ret_kurt = float(kurtosis(daily_returns.dropna().values, fisher=False))  # regular kurtosis

min_btl = minimum_backtest_length(
    observed_sharpe=obs_sharpe,
    n_trials=N_TRIALS,
    skewness=ret_skew,
    kurtosis=ret_kurt,
    confidence=0.95,
)
min_btl_years = min_btl / 252
available_years = n_days / 252

print('=' * 60)
print('DEFLATED SHARPE RATIO + MINIMUM BACKTEST LENGTH')
print('=' * 60)
print(f'  N trials tested:     {N_TRIALS}')
print(f'  Deflated Sharpe:     {dsr:.4f}')
print(f'  Gate (DSR > 0):      {"PASS" if dsr > 0 else "FAIL"}')
print(f'')
print(f'  Return skewness:     {ret_skew:.3f}')
print(f'  Return kurtosis:     {ret_kurt:.3f}')
print(f'  MinBTL:              {min_btl} days ({min_btl_years:.1f} years)')
print(f'  Available data:      {n_days} days ({available_years:.1f} years)')
print(f'  Gate (MinBTL < data): {"PASS" if min_btl < n_days else "FAIL"}')

DEFLATED SHARPE RATIO + MINIMUM BACKTEST LENGTH
  N trials tested:     1
  Deflated Sharpe:     1.0000
  Gate (DSR > 0):      PASS

  Return skewness:     -0.181
  Return kurtosis:     14.565
  MinBTL:              2982 days (11.8 years)
  Available data:      3460 days (13.7 years)
  Gate (MinBTL < data): PASS


In [9]:
# Cell 11: Walk-Forward Analysis
# Using the vectorized PortfolioBuilder approach for consistency
# 5 windows: 48-month train, 12-month test, rolling 12-month step

TRAIN_MONTHS = 48
TEST_MONTHS = 12
TRAIN_DAYS = TRAIN_MONTHS * 21  # ~1008 trading days
TEST_DAYS = TEST_MONTHS * 21    # ~252 trading days

# Get valid data range (post-warmup)
valid_prices = prices.iloc[WARMUP:]
valid_alpha = alpha
valid_dates = valid_prices.index

wf_results = []
wf_idx = 0
step = TEST_DAYS  # Non-overlapping test windows

print('Walk-Forward Analysis: 48-month train / 12-month test')
print('=' * 80)

train_start_loc = 0
while True:
    train_end_loc = train_start_loc + TRAIN_DAYS
    test_start_loc = train_end_loc
    test_end_loc = test_start_loc + TEST_DAYS
    
    if test_end_loc >= len(valid_dates):
        break
    
    train_start = str(valid_dates[train_start_loc].date())
    train_end = str(valid_dates[train_end_loc].date())
    test_start = str(valid_dates[test_start_loc].date())
    test_end = str(valid_dates[test_end_loc].date())
    
    # Build a fresh PortfolioBuilder for this window
    wf_config = PortfolioConfig(
        universe=list(prices.columns),
        signals=[],
        optimization='mean_variance',
        risk_aversion=RISK_AVERSION,
        max_weight=MAX_WEIGHT,
        min_weight=MIN_WEIGHT,
        target_gross=1.0,
        rebalance_frequency='2M',
        turnover_penalty=0.5,
        initial_cash=100000,
        commission=0.001,
    )
    wf_builder = PortfolioBuilder(config=wf_config)
    wf_builder.prices = prices  # Use full price history for signal computation
    wf_builder.signals = {'vol_scaled_momentum': alpha}
    
    # Set initial equal weights -- dynamic_reoptimize handles the rest
    wf_builder.weights = pd.Series(1.0 / n_assets, index=prices.columns)
    
    # Backtest on TEST period only
    wf_result = wf_builder.backtest(
        start_date=test_start,
        end_date=test_end,
        cost_model=cost_model,
        dynamic_reoptimize=True,
    )
    
    if wf_result and wf_result.get('n_days', 0) > 20:
        wf_sharpe = wf_result['sharpe_ratio']
        wf_return = wf_result['annualized_return']
        wf_dd = wf_result['max_drawdown']
        
        wf_results.append({
            'window': wf_idx + 1,
            'train_start': train_start,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end,
            'oos_sharpe': wf_sharpe,
            'oos_return': wf_return,
            'oos_max_dd': wf_dd,
            'oos_days': wf_result['n_days'],
        })
        
        print(f'  Window {wf_idx+1}: Train {train_start} to {train_end} | '
              f'Test {test_start} to {test_end} | '
              f'OOS Sharpe={wf_sharpe:.3f} | OOS Return={wf_return:.2%} | OOS DD={wf_dd:.2%}')
    
    wf_idx += 1
    train_start_loc += step

# Summary
wf_df = pd.DataFrame(wf_results)
if not wf_df.empty:
    wf_hit_rate = (wf_df['oos_sharpe'] > 0).mean()
    wf_avg_sharpe = wf_df['oos_sharpe'].mean()
    wf_avg_return = wf_df['oos_return'].mean()
    wf_sharpe_std = wf_df['oos_sharpe'].std()
    
    print(f'\n{"=" * 60}')
    print(f'WALK-FORWARD SUMMARY ({len(wf_df)} windows)')
    print(f'{"=" * 60}')
    print(f'  Hit rate (OOS Sharpe > 0): {wf_hit_rate:.1%}')
    print(f'  Avg OOS Sharpe:            {wf_avg_sharpe:.3f}')
    print(f'  Std OOS Sharpe:            {wf_sharpe_std:.3f}')
    print(f'  Avg OOS Return:            {wf_avg_return:.2%}')
    print(f'  Gate (hit rate > 55%):     {"PASS" if wf_hit_rate > 0.55 else "FAIL"}')
else:
    wf_hit_rate = 0
    wf_avg_sharpe = 0
    print('WARNING: No walk-forward windows completed.')

Walk-Forward Analysis: 48-month train / 12-month test


  Window 1: Train 2011-03-30 to 2015-04-02 | Test 2015-04-02 to 2016-04-04 | OOS Sharpe=0.268 | OOS Return=3.38% | OOS DD=-16.87%


  Window 2: Train 2012-03-29 to 2016-04-04 | Test 2016-04-04 to 2017-04-03 | OOS Sharpe=1.043 | OOS Return=11.20% | OOS DD=-6.20%


  Window 3: Train 2013-04-03 to 2017-04-03 | Test 2017-04-03 to 2018-04-04 | OOS Sharpe=1.515 | OOS Return=24.64% | OOS DD=-9.98%


  Window 4: Train 2014-04-02 to 2018-04-04 | Test 2018-04-04 to 2019-04-04 | OOS Sharpe=0.708 | OOS Return=10.80% | OOS DD=-19.97%


  Window 5: Train 2015-04-02 to 2019-04-04 | Test 2019-04-04 to 2020-04-03 | OOS Sharpe=-0.111 | OOS Return=-8.32% | OOS DD=-30.72%


  Window 6: Train 2016-04-04 to 2020-04-03 | Test 2020-04-03 to 2021-04-06 | OOS Sharpe=2.428 | OOS Return=66.74% | OOS DD=-9.84%


  Window 7: Train 2017-04-03 to 2021-04-06 | Test 2021-04-06 to 2022-04-04 | OOS Sharpe=1.628 | OOS Return=24.52% | OOS DD=-8.81%


  Window 8: Train 2018-04-04 to 2022-04-04 | Test 2022-04-04 to 2023-04-05 | OOS Sharpe=0.195 | OOS Return=1.92% | OOS DD=-14.41%


  Window 9: Train 2019-04-04 to 2023-04-05 | Test 2023-04-05 to 2024-04-08 | OOS Sharpe=1.984 | OOS Return=23.70% | OOS DD=-7.12%

WALK-FORWARD SUMMARY (9 windows)
  Hit rate (OOS Sharpe > 0): 88.9%
  Avg OOS Sharpe:            1.073
  Std OOS Sharpe:            0.875
  Avg OOS Return:            17.62%
  Gate (hit rate > 55%):     PASS


In [10]:
# Cell 12: Cost Sensitivity (1x, 1.5x, 2x, 3x costs)

cost_multipliers = [0, 1.0, 1.5, 2.0, 3.0]
cost_results = []

print('COST SENSITIVITY ANALYSIS')
print('=' * 70)
print(f'{"Multiplier":<12} {"Cost (bps)":<12} {"Sharpe":<10} {"Ann Return":<14} {"Max DD":<10}')
print('-' * 70)

for mult in cost_multipliers:
    if mult == 0:
        cm = None  # No costs
    else:
        cm = CompositeCostModel(
            models=(
                FixedCostModel(cost_per_trade=0.005 * mult),
                ProportionalCostModel(cost_bps=PROP_COST_BPS * mult),
            )
        )
    
    # Fresh builder for each cost level
    cs_config = PortfolioConfig(
        universe=list(prices.columns),
        signals=[],
        optimization='mean_variance',
        risk_aversion=RISK_AVERSION,
        max_weight=MAX_WEIGHT,
        min_weight=MIN_WEIGHT,
        target_gross=1.0,
        rebalance_frequency='2M',
        turnover_penalty=0.5,
        initial_cash=100000,
        commission=0.001,
    )
    cs_builder = PortfolioBuilder(config=cs_config)
    cs_builder.prices = prices
    cs_builder.signals = {'vol_scaled_momentum': alpha}
    # Equal-weight initial; dynamic reoptimize handles the rest
    cs_builder.weights = pd.Series(1.0 / n_assets, index=prices.columns)
    
    cs_result = cs_builder.backtest(
        start_date=backtest_start,
        end_date=END_DATE,
        cost_model=cm,
        dynamic_reoptimize=True,
    )
    
    cs_sharpe = cs_result.get('sharpe_ratio', 0)
    cs_ann_ret = cs_result.get('annualized_return', 0)
    cs_dd = cs_result.get('max_drawdown', 0)
    effective_bps = PROP_COST_BPS * mult if mult > 0 else 0
    
    cost_results.append({
        'multiplier': mult,
        'cost_bps': effective_bps,
        'sharpe': cs_sharpe,
        'ann_return': cs_ann_ret,
        'max_dd': cs_dd,
    })
    
    label = f'{mult}x' if mult > 0 else 'No cost'
    print(f'{label:<12} {effective_bps:<12.0f} {cs_sharpe:<10.3f} {cs_ann_ret:<14.2%} {cs_dd:<10.2%}')

cost_df = pd.DataFrame(cost_results)

# Gate checks
sharpe_2x = cost_df.loc[cost_df['multiplier'] == 2.0, 'sharpe'].iloc[0]
sharpe_3x = cost_df.loc[cost_df['multiplier'] == 3.0, 'sharpe'].iloc[0]

print(f'\nSharpe at 2x costs: {sharpe_2x:.3f} (gate: > 0) -> {"PASS" if sharpe_2x > 0 else "FAIL"}')
print(f'Sharpe at 3x costs: {sharpe_3x:.3f} (gate: > 0) -> {"PASS" if sharpe_3x > 0 else "FAIL"}')

COST SENSITIVITY ANALYSIS
Multiplier   Cost (bps)   Sharpe     Ann Return     Max DD    
----------------------------------------------------------------------


No cost      0            1.006      17.93%         -32.00%   


1.0x         10           0.885      15.42%         -32.00%   


1.5x         15           0.813      13.97%         -32.00%   


2.0x         20           0.740      12.53%         -32.00%   


3.0x         30           0.595      9.69%          -32.00%   

Sharpe at 2x costs: 0.740 (gate: > 0) -> PASS
Sharpe at 3x costs: 0.595 (gate: > 0) -> PASS


In [11]:
# Cell 13: Regime Analysis
from alpha_research.backtests.walkforward import RegimeAnalyzer

# Create market data from an equal-weighted portfolio of the universe
# (approximates the market)
market_returns = prices.pct_change().mean(axis=1)
market_cum = (1 + market_returns).cumprod()
market_df = pd.DataFrame({'close': market_cum}, index=prices.index)

# Regime analyzer expects equity_curve in the result dict
ra = RegimeAnalyzer(result, market_df)
regime_metrics = ra.analyze()

print('=' * 70)
print('REGIME ANALYSIS')
print('=' * 70)
print(f'{"Regime":<18} {"Sharpe":>8} {"Ann Return":>12} {"Max DD":>10} {"N Days":>8}')
print('-' * 70)

worst_regime_return = 0
worst_regime_name = ''

for regime_name, metrics in sorted(regime_metrics.items()):
    ann_r = metrics.get('annualized_return', 0)
    s = metrics.get('sharpe_ratio', 0)
    dd = metrics.get('max_drawdown', 0)
    nd = metrics.get('n_days', 0)
    
    print(f'{regime_name:<18} {s:>8.3f} {ann_r:>12.2%} {dd:>10.2%} {nd:>8}')
    
    if ann_r < worst_regime_return:
        worst_regime_return = ann_r
        worst_regime_name = regime_name

print(f'\nWorst regime: {worst_regime_name} ({worst_regime_return:.2%} ann)')
print(f'Gate (worst regime > -15%): {"PASS" if worst_regime_return > -0.15 else "FAIL"}')

REGIME ANALYSIS
Regime               Sharpe   Ann Return     Max DD   N Days
----------------------------------------------------------------------
trend_bear           -2.433      -52.69%    -88.64%      738
trend_bull            2.975       47.16%     -8.93%     2721
vol_high_vol          0.697       14.44%    -32.00%     1630
vol_low_vol           1.369       16.49%    -12.44%     1829

Worst regime: trend_bear (-52.69% ann)
Gate (worst regime > -15%): FAIL


In [12]:
# Cell 14: Parameter Sensitivity (+/-20% and +/-40%)

# Test sensitivity of three key parameters:
# 1. Volatility lookback (base=60)
# 2. Momentum lookback (base=252)
# 3. Risk aversion (base=3.0)

def run_sensitivity_backtest(vol_lb, mom_lb, risk_av):
    """Run a single backtest with varied parameters."""
    # Recompute signals with varied lookbacks
    vs = VolatilitySignal(lookback=vol_lb)
    ms = MomentumSignal(lookback=mom_lb, skip=21)
    
    rv = vs.compute(prices)
    rm = ms.compute(prices)
    trailing_v = prices.pct_change().rolling(mom_lb, min_periods=mom_lb).std() * np.sqrt(252)
    rm_sharpe = rm / trailing_v.replace(0, np.nan)
    rm_sharpe = rm_sharpe.replace([np.inf, -np.inf], np.nan)
    
    zv = expanding_zscore(rv)
    zm = expanding_zscore(rm_sharpe)
    zmr = z_mr  # Mean reversion doesn't change
    
    a = WEIGHT_INVVOL * zv + WEIGHT_MOMENTUM * zm + WEIGHT_MEANREV * zmr
    warmup_needed = max(WARMUP, mom_lb + MIN_PERIODS_ZSCORE)
    a = a.iloc[warmup_needed:]
    
    if a.empty or len(a) < 60:
        return np.nan, np.nan, np.nan
    
    sc = PortfolioConfig(
        universe=list(prices.columns), signals=[],
        optimization='mean_variance', risk_aversion=risk_av,
        max_weight=MAX_WEIGHT, min_weight=MIN_WEIGHT, target_gross=1.0,
        rebalance_frequency='2M', turnover_penalty=0.5,
        initial_cash=100000, commission=0.001,
    )
    sb = PortfolioBuilder(config=sc)
    sb.prices = prices
    sb.signals = {'vsm': a}
    sb.weights = pd.Series(1.0 / n_assets, index=prices.columns)
    
    bs = str(a.index[0].date())
    sr = sb.backtest(start_date=bs, end_date=END_DATE, cost_model=cost_model, dynamic_reoptimize=True)
    return sr.get('sharpe_ratio', 0), sr.get('annualized_return', 0), sr.get('max_drawdown', 0)

# Parameter variations
param_tests = [
    # (label, vol_lb, mom_lb, risk_av)
    ('Vol LB -40%', 36, 252, 3.0),
    ('Vol LB -20%', 48, 252, 3.0),
    ('BASE', 60, 252, 3.0),
    ('Vol LB +20%', 72, 252, 3.0),
    ('Vol LB +40%', 84, 252, 3.0),
    ('Mom LB -40%', 60, 151, 3.0),
    ('Mom LB -20%', 60, 202, 3.0),
    ('Mom LB +20%', 60, 302, 3.0),
    ('Mom LB +40%', 60, 353, 3.0),
    ('RiskAv -40%', 60, 252, 1.8),
    ('RiskAv -20%', 60, 252, 2.4),
    ('RiskAv +20%', 60, 252, 3.6),
    ('RiskAv +40%', 60, 252, 4.2),
]

print('PARAMETER SENSITIVITY ANALYSIS')
print('=' * 70)
print(f'{"Variation":<18} {"Vol LB":>8} {"Mom LB":>8} {"Risk Av":>8} {"Sharpe":>8} {"Return":>10} {"Max DD":>10}')
print('-' * 70)

sensitivity_results = []
for label, vlb, mlb, rav in param_tests:
    try:
        s, r, d = run_sensitivity_backtest(vlb, mlb, rav)
        sensitivity_results.append({'label': label, 'sharpe': s})
        if not np.isnan(s):
            print(f'{label:<18} {vlb:>8} {mlb:>8} {rav:>8.1f} {s:>8.3f} {r:>10.2%} {d:>10.2%}')
        else:
            print(f'{label:<18} {vlb:>8} {mlb:>8} {rav:>8.1f} {"N/A":>8}')
    except Exception as e:
        print(f'{label:<18} ERROR: {str(e)[:40]}')
        sensitivity_results.append({'label': label, 'sharpe': np.nan})

# Check sensitivity: Sharpe range within +/-20% variations
base_sharpe = [x['sharpe'] for x in sensitivity_results if x['label'] == 'BASE'][0]
pm20_sharpes = [x['sharpe'] for x in sensitivity_results 
                if '20%' in x['label'] and not np.isnan(x['sharpe'])]
if pm20_sharpes and base_sharpe > 0:
    sharpe_range = max(pm20_sharpes) - min(pm20_sharpes)
    sensitivity_pct = sharpe_range / abs(base_sharpe)
    print(f'\nSharpe range (+/-20%): {min(pm20_sharpes):.3f} to {max(pm20_sharpes):.3f}')
    print(f'Sensitivity: {sensitivity_pct:.1%} {"-- SENSITIVE" if sensitivity_pct > 0.5 else "-- ROBUST"}')

PARAMETER SENSITIVITY ANALYSIS
Variation            Vol LB   Mom LB  Risk Av   Sharpe     Return     Max DD
----------------------------------------------------------------------


Vol LB -40%              36      252      3.0    0.889     15.18%    -27.49%


Vol LB -20%              48      252      3.0    0.844     14.40%    -27.59%


BASE                     60      252      3.0    0.885     15.42%    -32.00%


Vol LB +20%              72      252      3.0    0.851     14.80%    -31.77%


...

Mom LB +40%              60      353      3.0    1.019     18.19%    -32.01%


RiskAv -40%              60      252      1.8    0.885     15.42%    -32.00%


RiskAv -20%              60      252      2.4    0.885     15.42%    -32.00%


RiskAv +20%              60      252      3.6    0.884     15.40%    -32.00%


RiskAv +40%              60      252      4.2    0.883     15.38%    -32.00%

Sharpe range (+/-20%): 0.844 to 0.984
Sensitivity: 15.7% -- ROBUST


In [13]:
# Cell 15: Decay & Capacity Analysis
from alpha_research.backtests.stats.decay_analysis import rolling_sharpe, strategy_half_life, sharpe_decay_rate

# Rolling Sharpe (3-month and 12-month windows)
roll_sr_3m = rolling_sharpe(daily_returns, window=63)
roll_sr_12m = rolling_sharpe(daily_returns, window=252)

# Strategy half-life
hl = strategy_half_life(roll_sr_12m)
hl_years = hl / 252 if hl is not None else None

# Sharpe decay rate
decay = sharpe_decay_rate(roll_sr_12m, window=63)

print('=' * 60)
print('DECAY & CAPACITY ANALYSIS')
print('=' * 60)
print(f'  Rolling Sharpe (3M) mean:    {roll_sr_3m.mean():.3f}')
print(f'  Rolling Sharpe (12M) mean:   {roll_sr_12m.mean():.3f}')
print(f'  Rolling Sharpe (12M) std:    {roll_sr_12m.std():.3f}')
print(f'')

if hl_years is not None:
    print(f'  Strategy half-life:          {hl:.0f} days ({hl_years:.1f} years)')
    print(f'  Gate (half-life > 2 yrs):    {"PASS" if hl_years > 2 else "FAIL"}')
else:
    print(f'  Strategy half-life:          N/A (no meaningful decay detected)')
    print(f'  Gate (half-life > 2 yrs):    PASS (no decay = infinite half-life)')
    hl_years = float('inf')  # No decay means edge persists

print(f'')
print(f'  Avg decay rate (last 63d):   {decay.dropna().iloc[-63:].mean():.6f} Sharpe/day')

# Plot rolling Sharpe
fig, ax = plt.subplots(figsize=(14, 5))
roll_sr_3m.plot(ax=ax, alpha=0.5, label='3-Month Rolling Sharpe', color='blue')
roll_sr_12m.plot(ax=ax, alpha=0.8, label='12-Month Rolling Sharpe', color='darkblue', linewidth=2)
ax.axhline(0, color='red', linewidth=0.5, linestyle='--')
ax.axhline(sharpe, color='green', linewidth=0.5, linestyle='--', label=f'Full-period Sharpe ({sharpe:.2f})')
ax.set_title(f'{STRATEGY_NAME} -- Rolling Sharpe Ratio')
ax.set_ylabel('Annualized Sharpe')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT + '/research/strategies/vol_scaled_momentum_2026-03-13_conditional/rolling_sharpe.png', dpi=100)
plt.show()

DECAY & CAPACITY ANALYSIS
  Rolling Sharpe (3M) mean:    1.335
  Rolling Sharpe (12M) mean:   1.114
  Rolling Sharpe (12M) std:    0.761

  Strategy half-life:          2937 days (11.7 years)
  Gate (half-life > 2 yrs):    PASS

  Avg decay rate (last 63d):   0.006537 Sharpe/day


## 16. Summary — Quantitative Gates

Results will be filled in after execution. See the code cell below for the computed gate table.

In [14]:
# Cell 16: Summary Gate Table

# Compute all gate values
gate_sharpe_is = sharpe
gate_dsr = dsr
gate_psr = psr
gate_wf_hit = wf_hit_rate if 'wf_hit_rate' in dir() else 0
gate_2x_cost = sharpe_2x if 'sharpe_2x' in dir() else 0
gate_3x_cost = sharpe_3x if 'sharpe_3x' in dir() else 0
gate_worst_regime = worst_regime_return if 'worst_regime_return' in dir() else -1
gate_half_life = hl_years if hl_years is not None else float('inf')
gate_min_btl = min_btl if 'min_btl' in dir() else 1e6
gate_max_dd = max_dd
gate_turnover = annual_turnover if 'annual_turnover' in dir() else 0

# Gate table
gates = [
    ('Sharpe (IS)', f'{gate_sharpe_is:.3f}', '> 0.5', 'PASS' if gate_sharpe_is > 0.5 else 'FAIL'),
    ('Deflated Sharpe', f'{gate_dsr:.4f}', '> 0', 'PASS' if gate_dsr > 0 else 'FAIL'),
    ('PSR', f'{gate_psr:.1%}', '> 80%', 'PASS' if gate_psr > 0.80 else 'FAIL'),
    ('WF Hit Rate', f'{gate_wf_hit:.1%}', '> 55%', 'PASS' if gate_wf_hit > 0.55 else 'FAIL'),
    ('Survives 2x Costs', f'{gate_2x_cost:.3f}', 'Sharpe > 0', 'PASS' if gate_2x_cost > 0 else 'FAIL'),
    ('Cost Sensitivity (3x)', f'{gate_3x_cost:.3f}', 'Sharpe > 0', 'PASS' if gate_3x_cost > 0 else 'FAIL'),
    ('Worst Regime Loss', f'{gate_worst_regime:.2%}', '> -15%', 'PASS' if gate_worst_regime > -0.15 else 'FAIL'),
    ('Strategy Half-Life', f'{gate_half_life:.1f} yrs' if gate_half_life != float('inf') else 'No decay', '> 2 yrs', 'PASS' if gate_half_life > 2 else 'FAIL'),
    ('MinBTL', f'{gate_min_btl/252:.1f} yrs', f'< {available_years:.1f} yrs', 'PASS' if gate_min_btl < n_days else 'FAIL'),
    ('Max Drawdown', f'{gate_max_dd:.2%}', '< 25%', 'PASS' if abs(gate_max_dd) < 0.25 else 'FAIL'),
    ('Annual Turnover', f'{gate_turnover:.0f}%', '< 150%', 'PASS' if gate_turnover < 150 else 'FAIL'),
]

print('=' * 75)
print('QUANTITATIVE GATES SUMMARY')
print('=' * 75)
print(f'{"Gate":<25} {"Value":<20} {"Threshold":<15} {"Status":<8}')
print('-' * 75)

n_pass = 0
n_total = len(gates)
for gate_name, value, threshold, status in gates:
    marker = '[PASS]' if status == 'PASS' else '[FAIL]'
    print(f'{gate_name:<25} {value:<20} {threshold:<15} {marker}')
    if status == 'PASS':
        n_pass += 1

print(f'\n{"=" * 75}')
print(f'GATES PASSED: {n_pass} / {n_total}')
print(f'{"=" * 75}')

print(f'\n--- Researcher Self-Assessment ---')
print(f'Strategy: {STRATEGY_NAME}')
print(f'The vol-scaled momentum strategy combines two academically robust factors')
print(f'(low-vol and momentum) with a mean-reversion dampener to reduce crash risk.')
print(f'The strategy uses pre-committed weights, expanding-window normalization,')
print(f'and dynamic weight re-optimization to avoid look-ahead bias.')
print(f'')
print(f'Confidence: MODERATE. The academic evidence is strong, but the 30-stock')
print(f'universe is small for cross-sectional strategies. The bi-monthly rebalance')
print(f'helps reduce turnover but may miss short-term momentum shifts.')
print(f'')
print(f'Top 3 Risks:')
print(f'1. Momentum crash risk during sharp market reversals (e.g., 2020 COVID)')
print(f'2. Small universe (30 stocks) may cause sector concentration')
print(f'3. Factor crowding -- low-vol and momentum are widely known factors')
print(f'')
print(f'Suggested PM Focus Areas:')
print(f'1. Walk-forward OOS Sharpe stability and hit rate')
print(f'2. Regime analysis during bear markets')
print(f'3. Sensitivity to volatility lookback parameter')

QUANTITATIVE GATES SUMMARY
Gate                      Value                Threshold       Status  
---------------------------------------------------------------------------
Sharpe (IS)               0.885                > 0.5           [PASS]
Deflated Sharpe           1.0000               > 0             [PASS]
PSR                       99.9%                > 80%           [PASS]
WF Hit Rate               88.9%                > 55%           [PASS]
Survives 2x Costs         0.740                Sharpe > 0      [PASS]
Cost Sensitivity (3x)     0.595                Sharpe > 0      [PASS]
Worst Regime Loss         -52.69%              > -15%          [FAIL]
Strategy Half-Life        11.7 yrs             > 2 yrs         [PASS]
MinBTL                    11.8 yrs             < 13.7 yrs      [PASS]
Max Drawdown              -32.00%              < 25%           [FAIL]
Annual Turnover           283%                 < 150%          [FAIL]

GATES PASSED: 8 / 11

--- Researcher Self-Assessment -